In [3]:
import os, json, yaml, logging, random, time, re
import nltk, pandas as pd
from nltk.util import ngrams
from collections import Counter
import google.generativeai as genai
from google.api_core import exceptions as google_exceptions


logging.basicConfig(format='%(asctime)s %(levelname)s: %(message)s', level=logging.INFO)


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

ModuleNotFoundError: No module named 'yaml'

In [ ]:
INPUT_FILE = ''
OUTPUT_FILE = 'linter_results.xlsx'
ENSEMBLE_CALLS = 1

# Gemini API
CFG = {
    'gemini_model': 'gemini-2.5-flash',
    'api_key': '',
    'max_retries': 3,
    'ngram_n': 3,
    'max_words': 150,
    'diversity_threshold': 0.5,
    'novelty_threshold': 0.2,
    'semantic_threshold': 0.2,
    'golden_prompts_path': 'golden_prompts.json'
    }

# Load golden prompts for novelty / semantic (token overlap) checks
if os.path.exists(CFG['golden_prompts_path']):
    golden_prompts = json.load(open(CFG['golden_prompts_path']))
else:
    golden_prompts = []

# Configure Gemini
genai.configure(api_key=CFG['api_key'])
gemini_model = genai.GenerativeModel(CFG['gemini_model'])

SAFETY_SETTINGS = {k: "BLOCK_NONE" for k in [
    "HARM_CATEGORY_HARASSMENT","HARM_CATEGORY_HATE_SPEECH",
    "HARM_CATEGORY_SEXUALLY_EXPLICIT","HARM_CATEGORY_DANGEROUS_CONTENT"]}

In [ ]:
def generate_with_retry(model, prompt, retries=CFG['max_retries']):
    for attempt in range(retries):
        try:
            return model.generate_content(prompt, safety_settings=SAFETY_SETTINGS)
        except google_exceptions.ServiceUnavailable:
            wait = (2**attempt) + random.random()
            logging.warning(f"503 error, retry in {wait:.1f}s…")
            time.sleep(wait)
        except Exception as e:
            logging.error(f"Fatal Gemini error: {e}")
            break
    return None

In [ ]:
EVALUATOR_SI = (
    "You are a Prompt Quality Assurance (PQA) Linter.\n"
    "Return exact JSON with keys prompt, score (1‑100), issues (list).\n"
    "Each issue item must include dimension, problem, suggestion.\n"
    "Dimensions to check: Clarity, Completeness, Constraint Adherence, Efficiency, Repetitiveness, Human‑likeness, Accuracy.\n"
    "If a dimension has no problems, omit it from the issues list. Score must reflect the number of PASS dimensions."
)

In [ ]:
def word_count(text):
    return len(nltk.word_tokenize(text))

# Novelty via n‑gram overlap
GSET = {g for t in golden_prompts for g in ngrams(nltk.word_tokenize(t.lower()), CFG['ngram_n'])}

def novelty_score(p):
    grams = list(ngrams(nltk.word_tokenize(p.lower()), CFG['ngram_n']))
    overlap = sum(1 for g in grams if g in GSET)
    return 1 - overlap / max(1, len(grams))

def lexical_diversity(p):
    toks = nltk.word_tokenize(p.lower())
    return len(set(toks)) / len(toks) if toks else 0

def semantic_jaccard(p):
    if not golden_prompts:
        return 1.0
    p_set = set(nltk.word_tokenize(p.lower()))
    sims = [len(p_set & set(nltk.word_tokenize(g.lower()))) / len(p_set | set(nltk.word_tokenize(g.lower()))) for g in golden_prompts]

In [ ]:
import json

def parse_llm(text: str):
    """Strip ```json fences and load."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"```json|```", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except Exception as e:
        logging.error(f"LLM JSON parse error: {e}")
        return {"prompt": "", "score": 0, "issues": []}

def score_gemini(prompt):
    payload = EVALUATOR_SI + "\nPrompt: " + prompt
    resp = generate_with_retry(gemini_model, payload)
    if resp and hasattr(resp, 'text'):
        return parse_llm(resp.text)
    return {"prompt": prompt, "score": 0, "issues": []}

In [ ]:
from tqdm import tqdm

# Load prompts
df = (pd.read_excel(INPUT_FILE) if INPUT_FILE and os.path.exists(INPUT_FILE) and INPUT_FILE.endswith('xlsx') else
      pd.read_csv(INPUT_FILE) if INPUT_FILE and os.path.exists(INPUT_FILE) else
      pd.DataFrame([{'prompt': input("Enter prompt: ")}]))

df.columns = df.columns.str.lower().str.strip()

# How many Gemini passes
if ENSEMBLE_CALLS is None:
    ENSEMBLE_CALLS = max(1, min(int(input("Gemini calls (1‑3): ") or 1), 3))

results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt_text = row['prompt']

    # ── LLM checks ─────────────────────────────────────
    llm_responses = [score_gemini(prompt_text) for _ in range(ENSEMBLE_CALLS)]
    best_llm = max(llm_responses, key=lambda d: d.get('score', 0))

    core_dims = ["Clarity","Completeness","Constraint Adherence","Efficiency",
                 "Repetitiveness","Human-likeness","Accuracy"]
    status_map = {d: 'PASS' for d in core_dims}
    for issue in best_llm.get('issues', []):
        status_map[issue['dimension']] = 'FAIL'

    # ── Classic metrics ────────────────────────────────
    wc = word_count(prompt_text)
    eff_pass = 'PASS' if wc <= CFG['max_words'] else 'FAIL'

    nov_val = novelty_score(prompt_text);   nov_pass = 'PASS' if nov_val >= CFG['novelty_threshold'] else 'FAIL'
    div_val = lexical_diversity(prompt_text); div_pass = 'PASS' if div_val >= CFG['diversity_threshold'] else 'FAIL'
    sem_val = semantic_jaccard(prompt_text);  sem_pass = 'PASS' if sem_val >= CFG['semantic_threshold'] else 'FAIL'

    # Classic dimension issues
    classic_issues = []
    if eff_pass == 'FAIL':
        classic_issues.append({"dimension":"Length","problem":f"{wc} words (>{CFG['max_words']})","suggestion":"Shorten to 150 words."})
    if nov_pass == 'FAIL':
        classic_issues.append({"dimension":"Novelty","problem":"High n‑gram overlap","suggestion":"Rewrite with new phrasing."})
    if div_pass == 'FAIL':
        classic_issues.append({"dimension":"Diversity","problem":"Low lexical variety","suggestion":"Use varied vocabulary."})
    if sem_pass == 'FAIL':
        classic_issues.append({"dimension":"Semantic","problem":"Too similar to existing prompts","suggestion":"Add new context."})

    # Total issues
    issues = best_llm.get('issues', []) + classic_issues

    # ── Compute final score ────────────────────────────
    pass_flags = list(status_map.values()) + [eff_pass, nov_pass, div_pass, sem_pass]
    score_pct = round((pass_flags.count('PASS') / len(pass_flags)) * 100)

    # ── Build record ──────────────────────────────────
    rec = {
        "prompt": prompt_text,
        "score": score_pct,
        "issues": issues,
    }
    # Add status fields
    for k, v in status_map.items():
        rec[f"{k}_status"] = v
    rec.update({
        "Length_status": eff_pass,
        "Novelty_status": nov_pass,
        "Diversity_status": div_pass,
        "Semantic_status": sem_pass,
    })

    results.append(rec)

out = pd.DataFrame(results)

In [ ]:
import json

for row in out.to_dict(orient="records"):
    print(json.dumps(row, indent=2, ensure_ascii=False))
    print("\n" + "-"*80 + "\n")
